# 08 - PySpark Time Buckets and Grouping


## Setup
Target: Configure Spark + JDBC helper.


In [3]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
spark = (SparkSession.builder.appName('capacity-demo').config('spark.driver.host','127.0.0.1').config('spark.driver.bindAddress','127.0.0.1').config('spark.jars.packages','org.postgresql:postgresql:42.7.4').getOrCreate())
JDBC_URL = 'jdbc:postgresql://host.docker.internal:5432/observability'
def load_query(q: str):
    return (spark.read.format('jdbc').option('url', JDBC_URL).option('dbtable', f'({q}) t').option('user','obs_user').option('password','obs_pass').option('driver','org.postgresql.Driver').load())


## Bucket + Group
Target: Build hourly/day aggregates in Spark.


In [4]:
q = """SELECT sampled_at, host, cpu_pct, mem_pct, region AS application, env AS service FROM lab.telemetry_cpu_raw WHERE sampled_at >= now() - interval '7 days'"""
df = load_query(q)
out = (df.withColumn('hour_bucket', F.date_trunc('hour', 'sampled_at')).withColumn('day_bucket', 
                            F.to_date('sampled_at')).groupBy('hour_bucket','day_bucket','host','application','service').agg(F.round(F.avg('cpu_pct'),2).alias('avg_cpu'), 
                            F.max('cpu_pct').alias('peak_cpu'), F.round(F.avg('mem_pct'),2).alias('avg_memory'), 
                            F.max('mem_pct').alias('peak_memory')).orderBy(F.col('hour_bucket').desc(), 'host'))
out.show(30, truncate=False)


+-------------------+----------+------+-----------+-------+-------+--------+----------+-----------+
|hour_bucket        |day_bucket|host  |application|service|avg_cpu|peak_cpu|avg_memory|peak_memory|
+-------------------+----------+------+-----------+-------+-------+--------+----------+-----------+
|2026-05-14 00:00:00|2026-05-14|host09|us-west-2  |stage  |49.27  |49.27   |46.12     |46.12      |
|2026-05-14 00:00:00|2026-05-14|host45|eu-west-1  |prod   |86.89  |86.89   |56.07     |56.07      |
|2026-05-14 00:00:00|2026-05-14|host50|eu-west-1  |stage  |71.97  |71.97   |75.36     |75.36      |
|2026-05-13 23:00:00|2026-05-13|host01|eu-west-1  |stage  |61.20  |61.20   |51.43     |51.43      |
|2026-05-13 23:00:00|2026-05-13|host01|us-east-1  |stage  |68.22  |71.90   |52.77     |54.91      |
|2026-05-13 23:00:00|2026-05-13|host02|eu-west-1  |stage  |58.22  |72.22   |49.78     |57.85      |
|2026-05-13 23:00:00|2026-05-13|host02|us-west-2  |prod   |81.85  |91.69   |52.80     |62.78      |
